<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

در این مثال فرض می‌کنیم سه Configuration را قبلاً روی یک Evaluation Dataset اجرا کرده‌ایم و نتایج واقعی ارزیابی آن‌ها را ثبت کرده‌ایم.

کد:
- نتایج را نگهداری می‌کند.
- امتیاز کیفیت را محاسبه می‌کند.
- Latency را نیز در رتبه‌بندی لحاظ می‌کند.
- بهترین Configuration را نمایش می‌دهد.

برای اینکه نتیجه آزمایش قابل اعتماد باشد:
1. تمام Configurationها را با یک Evaluation Dataset یکسان آزمایش کن.
2. تا جای ممکن فقط یک پارامتر را در هر آزمایش تغییر بده.
3. نتایج هر آزمایش را ذخیره کن تا امکان مقایسه در آینده وجود داشته باشد.
4. علاوه بر میانگین امتیازها، موارد شکست‌خورده و تغییرات عملکرد روی سؤال‌های مختلف را بررسی کن.
5. Latency و Cost را در کنار Quality Metrics در نظر بگیر.
6. اگر خروجی LLM متغیر است، آزمایش را چند بار تکرار کن و میزان نوسان را هم بررسی کن.

In [ ]:
# 1. Evaluation results from three RAG experiments
experiments = [
    {
        "name": "Configuration A",
        "precision": 0.70,
        "recall": 0.65,
        "faithfulness": 0.80,
        "answer_relevance": 0.85,
        "latency": 2.0
    },
    {
        "name": "Configuration B",
        "precision": 0.85,
        "recall": 0.90,
        "faithfulness": 0.92,
        "answer_relevance": 0.90,
        "latency": 4.0
    },
    {
        "name": "Configuration C",
        "precision": 0.80,
        "recall": 0.88,
        "faithfulness": 0.90,
        "answer_relevance": 0.88,
        "latency": 3.0
    }
]

# 2. Define evaluation weights
weights = {
    "precision": 0.15,
    "recall": 0.20,
    "faithfulness": 0.30,
    "answer_relevance": 0.25,
    "latency": 0.10
}

MAX_LATENCY = 5.0

# 3. Calculate overall score
def calculate_score(experiment):

    quality_score = (
        experiment["precision"] * weights["precision"]
        + experiment["recall"] * weights["recall"]
        + experiment["faithfulness"] * weights["faithfulness"]
        + experiment["answer_relevance"] * weights["answer_relevance"]
    )

    # Normalize latency to a score between 0 and 1
    latency_score = max(
        0,
        1 - experiment["latency"] / MAX_LATENCY
    )

    overall_score = (
        quality_score
        + latency_score * weights["latency"]
    )

    return overall_score


# 4. Rank configurations
for experiment in experiments:
    experiment["overall_score"] = calculate_score(experiment)

ranked_experiments = sorted(
    experiments,
    key=lambda item: item["overall_score"],
    reverse=True
)

# 5. Display comparison
print("===== RAG CONFIGURATION COMPARISON =====")

for rank, experiment in enumerate(ranked_experiments, start=1):

    print(f"\nRank: {rank}")
    print("Name:", experiment["name"])
    print(f"Precision: {experiment['precision']:.2f}")
    print(f"Recall: {experiment['recall']:.2f}")
    print(f"Faithfulness: {experiment['faithfulness']:.2f}")
    print(f"Answer Relevance: {experiment['answer_relevance']:.2f}")
    print(f"Latency: {experiment['latency']:.2f}s")
    print(f"Overall Score: {experiment['overall_score']:.3f}")

# 6. Best configuration
best = ranked_experiments[0]

print("\n===== BEST CONFIGURATION =====")
print(best["name"])
print(f"Overall Score: {best['overall_score']:.3f}")
